In [206]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
import datetime

In [218]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番','3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']
K_DCOLS = ['RaceID','3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']
DATE = "221124"
START_DATE = "221110"
END_DATE = "221123"

In [219]:
kdf

,着,艇番,選手登番,選手名,モーター,ボート,展示,進入,スタートタイミング,場所,...,日時,3連単_結果,3連単_払戻,3連複_結果,3連複_払戻,2連単_結果,2連単_払戻,2連複_結果,2連複_払戻,RaceID
0,1R,晴,西,1,1,福岡,2022_11_24,4-3-5,7900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1R,晴,西,1,1,福岡,2022_11_24,4-3-5,7900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1R,晴,西,1,1,福岡,2022_11_24,4-3-5,7900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1R,晴,西,1,1,福岡,2022_11_24,4-3-5,7900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1R,晴,西,1,1,福岡,2022_11_24,4-3-5,7900,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,12R,曇り,西,2,1,桐生,2022_11_24,3-2-4,37000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
775,12R,曇り,西,2,1,桐生,2022_11_24,3-2-4,37000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
776,12R,曇り,西,2,1,桐生,2022_11_24,3-2-4,37000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
777,12R,曇り,西,2,1,桐生,2022_11_24,3-2-4,37000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [217]:
k_files = glob.glob("../csv/K" + DATE + ".csv")
b_files = glob.glob("../csv/B" + DATE + ".csv")
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp
kdf = concat_files(k_files)
bdf = concat_files(b_files)
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid

df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)
zero = []

for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero
df_test = df_encoded.drop(['選手名','RaceID','着','3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻'],axis=1)

with open('../model/lgb_clf_tenji_nasi.pickle', 'rb') as f:
    lgb_clf = pickle.load(f)
df_test
df_pred = lgb_clf.predict(df_test ,num_iteration=lgb_clf.best_iteration)
df_encoded["pred"] = df_pred

pred_rank = pd.DataFrame()
for i in df_encoded["RaceID"].unique():
    prd = df_encoded[df_encoded["RaceID"] == i]["pred"].rank(ascending=False)
    pred_rank = pd.concat([pred_rank,prd])
df_encoded["pred_rank"] = pred_rank
places = df_encoded['場所'].unique()
place_list = {
            '桐生':'01',
            '戸田':'02',
            '江戸川':'03',
            '平和島':'04',
            '多摩川':'05',
            '浜名湖':'06',
            '蒲郡':'07',
            '常滑':'08',
            '津':'09',
            '三国':'10',
            'びわこ':'11',
            '住之江':'12',
            '尼崎':'13',
            '鳴門':'14',
            '丸亀':'15',
            '児島':'16',
            '宮島':'17',
            '徳山':'18',
            '下関':'19',
            '若松':'20',
            '芦屋':'21',
            '福岡':'22',
            '唐津':'23',
            '大村':'24'}
def get_keys_from_value(d, val):
    return [k for k, v in d.items() if v == val]

for p in places:
    if p < 10:
        v = '0' + str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    else:
        v = str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    os.makedirs('../prediction/' + DATE + '/',exist_ok=True)
    with open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w"):
        pass
    f = open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w")
    print(PLACE,file=f)
    y_predp = df_encoded[df_encoded['場所'] == p]
    r_list = y_predp['R'].unique()
    
    print('-----------------------',file=f)
    for i in r_list:
        print("第"+str(i) + "R",file=f)
        race_pd = y_predp[y_predp["R"] == i]
        first = race_pd[race_pd['pred_rank'] == 1]['艇番'].iloc[0]
        second = race_pd[race_pd['pred_rank'] == 2]['艇番'].iloc[0]
        third = race_pd[race_pd['pred_rank'] == 3]['艇番'].iloc[0]
        print(str(first)+'-'+str(second) +'-' + str(third),file=f)
    print('-----------------------\n',file=f)
    f.close()
odds_vali = df_encoded[['RaceID','3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']]
raceid = odds_vali['RaceID'].value_counts(sort =False).index

odds_vali_r = pd.DataFrame([],columns = odds_vali.columns)

for i,r in tqdm(enumerate(raceid)):
    o = odds_vali[odds_vali['RaceID'] == r][0:1]
    odds_vali_r = pd.concat([odds_vali_r, o])

group =  df_encoded['RaceID'].value_counts(sort = False)
pred = df_pred
y_vali = df_encoded['着']

def cal(y_vali,y_pred_model ,val_group, odds_vali_r):
    #Validデータ的中率の算出
    j = 0
    m = -1

    cols = ['2連単_払戻','3連単_払戻','3連複_払戻','予測','結果']
    prize = pd.DataFrame(np.zeros([val_group.shape[0],5]),columns=cols)

    solo_count = 0
    solo_prize = []

    doub_count = 0
    doub_prize = []

    tri_count = 0
    tri_prize = []

    trio_count = 0
    trio_prize = []

    odds_vali_r = odds_vali_r.astype(str)


    for i in tqdm(val_group):
        result = y_pred_model[j:j+i] #グループでの順位
        ans = y_vali.iloc[j:j+i].reset_index() #答え
        result1st = np.argmax(result) + 1 #予測の1位

        m = m + 1

        if len(np.where(result==sorted(result)[::-1][0])[0])>1:
            result2nd = np.where(result==sorted(result)[::-1][1])[0][0] + 1
            result3rd = np.where(result==sorted(result)[::-1][1])[0][1] + 1
        else:
            if i > 1:
                result2nd = np.where(result==sorted(result)[::-1][1])[0][0] + 1
            if i > 2:
                result3rd = np.where(result==sorted(result)[::-1][2])[0][0] + 1

        ans1st = int(ans[ans["着"]==ans['着'].min()].index.values[0]) + 1
        and_sort = ans
        and_sort.sort_values(["着"],ascending=True,inplace=True)
        if i > 2:
            a2n = int(and_sort.iloc[1][1])
            a3n = int(and_sort.iloc[2][1])
            
            if len(ans[ans["着"]==a2n].index.values)>1:
                ans2nd = int(ans[ans["着"]==a2n].index.values[0]) + 1
                ans3rd = int(ans[ans["着"]==a3n].index.values[0]) + 1
                print('i')
            else:
                if i > 1:
                    ans2nd = int(ans[ans["着"]==a2n].index.values[0]) +1
                if i > 2:
                    ans3rd = int(ans[ans["着"]==a3n].index.values[0]) +1
        ans.sort_index(inplace=True)
        
        
        if ans1st==result1st:
            #print(ans1st,result1st)
            solo_count = solo_count+1

        if i > 1:
            if (ans1st==result1st)&(ans2nd==result2nd):
                doub_count = doub_count+1
                if str.isdecimal(odds_vali_r.loc[:,'2連単_払戻'].iat[m]):
                    doub_p = int(odds_vali_r.loc[:,'2連単_払戻'].iat[m])
                    doub_prize.append(doub_p)
                    prize.iloc[m,0] = doub_p

        if i > 2:
            if (ans1st==result1st)&(ans2nd==result2nd)&(ans3rd==result3rd):
                tri_count = tri_count+1 
                if str.isdecimal(odds_vali_r.loc[:,'3連単_払戻'].iat[m]):
                    tri_p = int(odds_vali_r.loc[:,'3連単_払戻'].iat[m])
                    tri_prize.append(tri_p)
                    prize.iloc[m,1] = tri_p

            ans_set = set([ans1st, ans2nd, ans3rd])
            res_set = set([result1st, result2nd,result3rd])
            if ans_set == res_set:
                trio_count = trio_count+1
                if str.isdecimal(odds_vali_r.loc[:,'3連複_払戻'].iat[m]):
                    trio_p = int(odds_vali_r.loc[:,'3連複_払戻'].iat[m])
                    trio_prize.append(trio_p)
                    prize.iloc[m,2] = trio_p
        
        ans_str = str(ans1st) + '-' + str(ans2nd) + '-' + str(ans3rd)
        res_str = str(result1st) + '-' + str(result2nd) + '-' + str(result3rd)
        prize.iloc[m,4] = ans_str
        prize.iloc[m,3] = res_str
        j = j + i
    
    prize = pd.concat([prize,odds_vali_r['RaceID'].reset_index(drop = True)],axis=1)
    return solo_count, doub_count, doub_prize, tri_count, tri_prize, trio_count, trio_prize, len(val_group), prize

def sum(list):
    sum = 0
    for i in list:
        sum = sum + i
    return sum

solo_count, doub_count, doub_prize, tri_count, tri_prize, trio_count, trio_prize, count, prize = cal(y_vali,pred,group, odds_vali_r)

100%|██████████| 1/1 [00:00<00:00, 112.57it/s]


ValueError: You are trying to merge on object and float64 columns. If you wish to proceed you should use pd.concat

In [198]:
df_encoded.to_csv('df_encoded.csv')

In [204]:
f = open('pred_nasi.txt', 'w')

print('--------------------------',file = f)
print('DATE : ', DATE, file = f) 
print('購入金額', count*100,'円',file = f)
print('--------------------------',file = f)
print('単勝',file = f)
print("的中率：",round(solo_count/count*100,2),"%",file = f)
print('--------------------------',file = f)
print('2連単',file = f)
print("的中率：",round(doub_count/count*100,2),"%",file = f)
print("払戻金額-購入金額：",round(sum(doub_prize) - (count*100),2),"円",file = f)
print("回収率：",round((sum(doub_prize) / (count*100) *100) ,2),"%",file = f)
print('--------------------------',file = f)
print('3連単',file = f)
print("的中率：",round(tri_count/count*100,2),"%",file = f)
print("払戻金額-購入金額：",round(sum(tri_prize) - (count*100),2),"円",file = f)
print("回収率：",round((sum(tri_prize) / (count*100)*100),2),"%",file = f)
print('--------------------------',file = f)
print('3連複',file = f)
print("的中率：",round(trio_count/count*100,2),"%",file = f)
print("払戻金額-購入金額：",round(sum(trio_prize) - (count*100),2),"円",file = f)
print("回収率：",round((sum(trio_prize) / (count*100)*100),2),"%",file = f)
print('--------------------------',file = f)

f.close()

In [203]:
prize

,2連単_払戻,3連単_払戻,3連複_払戻,予測,結果,RaceID
0,0.0,0.0,0.0,1-2-4,1-6-2,2022-11-26-24-01
1,0.0,0.0,230.0,1-6-4,4-1-6,2022-11-26-24-02
2,0.0,0.0,0.0,1-4-2,1-3-2,2022-11-26-24-03
3,0.0,0.0,0.0,2-4-3,1-2-3,2022-11-26-24-04
4,0.0,0.0,0.0,2-5-1,4-5-6,2022-11-26-24-05
...,...,...,...,...,...,...
139,0.0,0.0,0.0,1-5-2,6-5-3,2022-11-26-01-08
140,0.0,0.0,170.0,1-2-4,1-4-2,2022-11-26-01-09
141,0.0,0.0,0.0,1-2-3,1-3-4,2022-11-26-01-10
142,270.0,0.0,0.0,1-3-2,1-3-5,2022-11-26-01-11


In [205]:
prize.to_csv('prize.csv')

In [185]:
prize['3連複_払戻'].sum()-prize.shape[0]*100

1780.0

In [186]:
prize.shape[0]

132